# Direction 9: Fine-tune a Small Query Rewriter

This notebook fine-tunes a small causal language model to replace the API-based `IntentExpander` for complex hotel-review query rewriting.

Expected input files generated by `generate_query_rewrite_dataset.py`:

- `supervised_data/query_rewrite_train.jsonl`
- `supervised_data/query_rewrite_val.jsonl`
- `supervised_data/query_rewrite_test.jsonl`

Main outputs:

- `finetune_outputs/query_rewriter_lora/`
- `finetune_outputs/eval_predictions.csv`
- `finetune_outputs/eval_summary.csv`


## 0. Optional Dependency Installation

Run this cell on the server only if the environment does not already have the required packages.

In [ ]:
# %pip install -U torch transformers datasets peft accelerate sentencepiece pandas

In [1]:
import os

os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""
os.environ["all_proxy"] = ""

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["OMP_NUM_THREADS"] = "8"

In [2]:
from pathlib import Path
import json
import time
import math

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel

BASE_DIR = Path.cwd()
if BASE_DIR.name != "task9_scripts_admit":
    BASE_DIR = Path("/root/task9_scripts_admit")  # Change this on the server if needed.

DATA_DIR = BASE_DIR / "supervised_data"
OUTPUT_DIR = BASE_DIR / "finetune_outputs"
ADAPTER_DIR = OUTPUT_DIR / "query_rewriter_lora"
PRED_PATH = OUTPUT_DIR / "eval_predictions.csv"
SUMMARY_PATH = OUTPUT_DIR / "eval_summary.csv"

# You may replace this with a local model path on the server.
# Qwen3-4B-Instruct is used as a locally fine-tunable query rewriter baseline.
MODEL_NAME_OR_PATH = "Qwen/Qwen3-4B-Instruct-2507"

MAX_LENGTH = 1024
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
RUN_BASE_MODEL_EVAL = False  # Set True if you want to compare the unfine-tuned small model.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("MODEL:", MODEL_NAME_OR_PATH)
print("CUDA available:", torch.cuda.is_available())


BASE_DIR: /root/task9_scripts_admit
DATA_DIR: /root/task9_scripts_admit/supervised_data
MODEL: Qwen/Qwen3-4B-Instruct-2507
CUDA available: True


In [3]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_records = load_jsonl(DATA_DIR / "query_rewrite_train.jsonl")
val_records = load_jsonl(DATA_DIR / "query_rewrite_val.jsonl")
test_records = load_jsonl(DATA_DIR / "query_rewrite_test.jsonl")

print("train:", len(train_records))
print("val:", len(val_records))
print("test:", len(test_records))
print(json.dumps(train_records[0], ensure_ascii=False, indent=2)[:1200])


train: 60
val: 7
test: 8
{
  "messages": [
    {
      "role": "system",
      "content": "你是酒店评论 RAG 系统的复杂 Query 改写标注员。\n\n任务：把用户的复杂问题改写为 1-3 个适合检索酒店评论的中文 Query，并为每个 Query 分配权重。\n\n要求：\n- 每个改写 Query 只聚焦一个清晰关注点。\n- 必须保留原问题中的房型、人群、时间、比较对象等关键约束。\n- 多意图问题应拆分为多个 Query；明确单意图问题不要强行拆分。\n- 不要加入用户没有表达、且无法从评论中检索验证的实时信息，如今日房态、当前价格、实时活动。\n- weight 只能是 0.2、0.4、0.6、0.8、1.0，且同一 user_query 下权重总和必须为 1.0。\n- 只输出 JSON，不要输出解释或 Markdown。\n"
    },
    {
      "role": "user",
      "content": "打算带家人度假，想了解这家酒店的花园景观是否值得期待，还有餐厅好不好吃。"
    },
    {
      "role": "assistant",
      "content": "{\"rewritten_queries\": [{\"query\": \"适合家庭度假的酒店，住客对整体度假体验的评价有哪些？\", \"weight\": 0.2}, {\"query\": \"酒店花园景观实际效果如何？带家人入住的客人对观景体验的描述有哪些？\", \"weight\": 0.4}, {\"query\": \"酒店餐厅菜品质量、服务态度和用餐环境等餐饮相关的真实住客评价\", \"weight\": 0.4}]}"
    }
  ],
  "metadata": {
    "source_qid": "16",
    "query_type": "场景型",
    "expected_intents": "度假;花园景观;餐饮"
  }
}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True,
    use_fast=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True,
    torch_dtype=dtype if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

def chat_text(messages: list[dict], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    text = ""
    for msg in messages:
        text += f"{msg['role']}: {msg['content']}\n"
    if add_generation_prompt:
        text += "assistant: "
    return text

print("tokenizer pad token:", tokenizer.pad_token)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Error while downloading from https://cas-bridge.xethub.hf.co/xet-bridge-us/6891e3bb084ce75acffb033d/3be27e0bee966a1147d96aaebaef4521a26627abef249998ebfade36c10408dd?Expires=1781010082&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY4OTFlM2JiMDg0Y2U3NWFjZmZiMDMzZC8zYmUyN2UwYmVlOTY2YTExNDdkOTZhYWViYWVmNDUyMWEyNjYyN2FiZWYyNDk5OThlYmZhZGUzNmMxMDQwOGRkKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MTAxMDA4Mn19fV19&Signature=MEQCIGCRav68QkQ6RqP%7EnWE9Wdonx5PsOCjbJZqcbI%7Ev5ClBAiBFUBw7uaRfqAtzsNEL0Aa1J8j70bbiYeoxX9EegFQAdg__&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=62171e3b6a99db28e0b3159d&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model-00002-of-00003.safetensors%3B+filename%3D%22model-00002-of-00003.safetensors%22%3B&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260609%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260609T120122Z&X-Amz-Expires=3600&X-Amz-SignedHeaders=host&X-Amz-Signa

model-00002-of-00003.safetensors:  80%|#######9  | 3.18G/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer pad token: <|endoftext|>


## 1. Optional Baseline Evaluation

This evaluates the base small model before fine-tuning. It is disabled by default to save time.

In [5]:
INTENT_ALIASES = {
    "亲子": ["亲子", "孩子", "小孩", "儿童", "家庭"],
    "交通": ["交通", "出行", "地铁", "打车", "位置"],
    "早餐": ["早餐", "早饭", "餐饮"],
    "噪音": ["噪音", "吵", "隔音", "安静", "睡眠"],
    "性价比": ["性价比", "值不值", "值得", "划算", "价格"],
    "服务": ["服务", "前台", "响应", "接待"],
    "卫生": ["卫生", "干净", "清洁", "床品", "卫生间"],
    "商务": ["商务", "出差", "办公", "会议"],
    "设施": ["设施", "设备", "装修"],
    "空间": ["空间", "面积", "宽敞", "小不小", "局促"],
    "最近": ["最近", "现在", "最新", "当前"],
}

def parse_json_response(text: str):
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise

def coverage_score(expected_intents: str, rewritten_text: str):
    labels = [x.strip() for x in str(expected_intents).split(";") if x.strip()]
    if not labels:
        return 0, 0, 0.0, ""
    matched = []
    for label in labels:
        aliases = INTENT_ALIASES.get(label, [label])
        if any(alias in rewritten_text for alias in aliases):
            matched.append(label)
    return len(matched), len(labels), round(len(matched) / len(labels), 4), ";".join(matched)

def normalize_rewrites(data):
    rewrites = data.get("rewritten_queries", []) if isinstance(data, dict) else []
    if not isinstance(rewrites, list):
        return [], False
    cleaned = []
    for item in rewrites[:3]:
        if not isinstance(item, dict):
            continue
        query = str(item.get("query", "")).strip()
        if not query:
            continue
        try:
            weight = float(item.get("weight", 0))
        except Exception:
            weight = 0.0
        cleaned.append({"query": query, "weight": weight})
    format_ok = 1 <= len(cleaned) <= 3 and abs(sum(x["weight"] for x in cleaned) - 1.0) < 1e-5
    return cleaned, format_ok

@torch.no_grad()
def generate_one(model, record, max_new_tokens=256):
    prompt_messages = record["messages"][:-1]
    prompt = chat_text(prompt_messages, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.time()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    latency = time.time() - start
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text, latency

def evaluate_model(model, records, version: str):
    model.eval()
    rows = []
    for i, record in enumerate(records, 1):
        expected_intents = record.get("metadata", {}).get("expected_intents", "")
        user_query = record["messages"][1]["content"]
        try:
            text, latency = generate_one(model, record)
            data = parse_json_response(text)
            rewrites, format_ok = normalize_rewrites(data)
            rewritten_text = " ".join(item["query"] for item in rewrites)
            matched_count, expected_count, coverage, matched_labels = coverage_score(expected_intents, rewritten_text)
            error = ""
        except Exception as exc:
            text, latency = "", math.nan
            rewrites, format_ok = [], False
            matched_count, expected_count, coverage, matched_labels = 0, len(str(expected_intents).split(";")), 0.0, ""
            error = repr(exc)
        rows.append({
            "version": version,
            "idx": i,
            "user_query": user_query,
            "expected_intents": expected_intents,
            "rewritten_queries": " ||| ".join(item["query"] for item in rewrites),
            "weights": ";".join(str(item["weight"]) for item in rewrites),
            "format_ok": format_ok,
            "matched_intents": matched_labels,
            "intent_coverage": coverage,
            "latency_seconds": round(latency, 4) if not math.isnan(latency) else None,
            "raw_output": text,
            "error": error,
        })
        print(f"{version} [{i}/{len(records)}] coverage={coverage}, format_ok={format_ok}")
    return pd.DataFrame(rows)


In [6]:
baseline_eval_df = None
if RUN_BASE_MODEL_EVAL:
    baseline_eval_df = evaluate_model(model, test_records, version="base_small_model")
    display(baseline_eval_df.head())


## 2. LoRA Fine-tuning

In [7]:
class SFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.features = []
        for record in records:
            text = chat_text(record["messages"], add_generation_prompt=False)
            encoded = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding=False,
            )
            self.features.append(encoded)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        item = self.features[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["input_ids"], dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(x["input_ids"].shape[0] for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for item in batch:
        pad_len = max_len - item["input_ids"].shape[0]
        input_ids.append(torch.cat([item["input_ids"], torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
    }

train_dataset = SFTDataset(train_records, tokenizer, MAX_LENGTH)
val_dataset = SFTDataset(val_records, tokenizer, MAX_LENGTH)
print("train dataset:", len(train_dataset))
print("val dataset:", len(val_dataset))


train dataset: 60
val dataset: 7


In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer_checkpoints"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)


The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [9]:
trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved LoRA adapter to:", ADAPTER_DIR)


Epoch,Training Loss,Validation Loss
1,2.160100,0.997925
2,0.527900,0.636986
3,0.380800,0.608592


Saved LoRA adapter to: /root/task9_scripts_admit/finetune_outputs/query_rewriter_lora


## 3. Evaluate Fine-tuned Model

In [12]:
from pathlib import Path
import json
import time
import math
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_DIR = Path("/root/task9_scripts_admit")
DATA_DIR = BASE_DIR / "supervised_data"
OUTPUT_DIR = BASE_DIR / "finetune_outputs"
ADAPTER_DIR = OUTPUT_DIR / "query_rewriter_lora"

MODEL_NAME_OR_PATH = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True,
    use_fast=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("Loaded fine-tuned LoRA model:", ADAPTER_DIR)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded fine-tuned LoRA model: /root/task9_scripts_admit/finetune_outputs/query_rewriter_lora


In [13]:
def chat_text(messages: list[dict], add_generation_prompt: bool = False) -> str:
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
    )

INTENT_ALIASES = {
    "亲子": ["亲子", "孩子", "小孩", "儿童", "家庭"],
    "交通": ["交通", "出行", "地铁", "打车", "位置"],
    "早餐": ["早餐", "早饭", "餐饮"],
    "噪音": ["噪音", "吵", "隔音", "安静", "睡眠"],
    "性价比": ["性价比", "值不值", "值得", "划算", "价格"],
    "服务": ["服务", "前台", "响应", "接待"],
    "卫生": ["卫生", "干净", "清洁", "床品", "卫生间"],
    "商务": ["商务", "出差", "办公", "会议"],
    "设施": ["设施", "设备", "装修"],
    "空间": ["空间", "面积", "宽敞", "小不小", "局促"],
    "最近": ["最近", "现在", "最新", "当前"],
}

def parse_json_response(text: str):
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise

def coverage_score(expected_intents: str, rewritten_text: str):
    labels = [x.strip() for x in str(expected_intents).split(";") if x.strip()]
    if not labels:
        return 0, 0, 0.0, ""
    matched = []
    for label in labels:
        aliases = INTENT_ALIASES.get(label, [label])
        if any(alias in rewritten_text for alias in aliases):
            matched.append(label)
    return len(matched), len(labels), round(len(matched) / len(labels), 4), ";".join(matched)

def normalize_rewrites(data):
    rewrites = data.get("rewritten_queries", []) if isinstance(data, dict) else []
    if not isinstance(rewrites, list):
        return [], False

    cleaned = []
    for item in rewrites[:3]:
        if not isinstance(item, dict):
            continue
        query = str(item.get("query", "")).strip()
        if not query:
            continue
        try:
            weight = float(item.get("weight", 0))
        except Exception:
            weight = 0.0
        cleaned.append({"query": query, "weight": weight})

    format_ok = 1 <= len(cleaned) <= 3 and abs(sum(x["weight"] for x in cleaned) - 1.0) < 1e-5
    return cleaned, format_ok

@torch.no_grad()
def generate_one(model, record, max_new_tokens=256):
    prompt_messages = record["messages"][:-1]
    prompt = chat_text(prompt_messages, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    latency = time.time() - start

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text, latency

def evaluate_model(model, records, version: str):
    model.eval()
    rows = []

    for i, record in enumerate(records, 1):
        expected_intents = record.get("metadata", {}).get("expected_intents", "")
        user_query = record["messages"][1]["content"]

        try:
            text, latency = generate_one(model, record)
            data = parse_json_response(text)
            rewrites, format_ok = normalize_rewrites(data)

            rewritten_text = " ".join(item["query"] for item in rewrites)
            matched_count, expected_count, coverage, matched_labels = coverage_score(
                expected_intents,
                rewritten_text,
            )
            error = ""
        except Exception as exc:
            text, latency = "", math.nan
            rewrites, format_ok = [], False
            coverage, matched_labels = 0.0, ""
            error = repr(exc)

        rows.append({
            "version": version,
            "idx": i,
            "user_query": user_query,
            "expected_intents": expected_intents,
            "rewritten_queries": " ||| ".join(item["query"] for item in rewrites),
            "weights": ";".join(str(item["weight"]) for item in rewrites),
            "format_ok": format_ok,
            "matched_intents": matched_labels,
            "intent_coverage": coverage,
            "latency_seconds": round(latency, 4) if not math.isnan(latency) else None,
            "raw_output": text,
            "error": error,
        })

        print(f"{version} [{i}/{len(records)}] coverage={coverage}, format_ok={format_ok}")

    return pd.DataFrame(rows)

In [14]:
COMPLEX_QUERY_PATH = DATA_DIR / "complex_queries.csv"
COMPLEX25_PRED_PATH = OUTPUT_DIR / "eval_complex25_predictions.csv"
COMPLEX25_SUMMARY_PATH = OUTPUT_DIR / "eval_complex25_summary.csv"

complex_df = pd.read_csv(COMPLEX_QUERY_PATH, encoding="utf-8")

# 用和训练数据一致的 system prompt
train_path = DATA_DIR / "query_rewrite_train.jsonl"
with train_path.open("r", encoding="utf-8") as f:
    first_train_record = json.loads(f.readline())
system_prompt = first_train_record["messages"][0]["content"]

complex_records = []
for _, row in complex_df.iterrows():
    complex_records.append({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row["query"]},
            {"role": "assistant", "content": ""}
        ],
        "metadata": {
            "source_qid": str(row["qid"]),
            "query_type": row["type"],
            "expected_intents": row["expected_intents"],
        }
    })

complex25_eval_df = evaluate_model(
    model,
    complex_records,
    version="finetuned_lora_complex25"
)

complex25_eval_df.to_csv(COMPLEX25_PRED_PATH, index=False, encoding="utf-8-sig")
print("Saved complex25 predictions:", COMPLEX25_PRED_PATH)

complex25_summary_df = (
    complex25_eval_df.groupby("version", as_index=False)
    .agg(
        num_examples=("idx", "count"),
        parse_success_rate=("format_ok", "mean"),
        mean_intent_coverage=("intent_coverage", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
    )
)

for col in ["parse_success_rate", "mean_intent_coverage", "mean_latency_seconds"]:
    complex25_summary_df[col] = complex25_summary_df[col].round(4)

complex25_summary_df.to_csv(COMPLEX25_SUMMARY_PATH, index=False, encoding="utf-8-sig")
display(complex25_summary_df)

finetuned_lora_complex25 [1/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [2/25] coverage=0.75, format_ok=True
finetuned_lora_complex25 [3/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [4/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [5/25] coverage=0.75, format_ok=True
finetuned_lora_complex25 [6/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [7/25] coverage=0.5, format_ok=True
finetuned_lora_complex25 [8/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [9/25] coverage=0.6667, format_ok=True
finetuned_lora_complex25 [10/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [11/25] coverage=0.75, format_ok=True
finetuned_lora_complex25 [12/25] coverage=0.6667, format_ok=True
finetuned_lora_complex25 [13/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [14/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [15/25] coverage=1.0, format_ok=True
finetuned_lora_complex25 [16/25] coverage=1.0, format_ok=True
finetune

,version,num_examples,parse_success_rate,mean_intent_coverage,mean_latency_seconds
0,finetuned_lora_complex25,25,1.0,0.8053,3.5087
